In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.metrics import accuracy_score, f1_score
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC
from sklearn.feature_extraction.text import TfidfVectorizer
from konlpy.tag import Komoran

# 자연어 처리 순서
1. 데이터의 로드
2. 데이터 튜닝
3. 데이터 분할
4. 토큰화
5. 벡터화
6. 모델 학습
7. 평가

In [2]:
# 데이터를 로드
df = pd.read_csv("../data/ratings_train.txt", sep = '\t')
df.head()

,id,document,label
0,9976970,아 더빙.. 진짜 짜증나네요 목소리,0
1,3819312,흠...포스터보고 초딩영화줄....오버연기조차 가볍지 않구나,1
2,10265843,너무재밓었다그래서보는것을추천한다,0
3,9045019,교도소 이야기구먼 ..솔직히 재미는 없다..평점 조정,0
4,6483659,사이몬페그의 익살스런 연기가 돋보였던 영화!스파이더맨에서 늙어보이기만 했던 커스틴 ...,1


In [3]:
# 필요 없는 컬럼을 제외 -> id 제외 -> id에 중복 값이 존재하지 않는다.
df.drop('id', axis=1, inplace=True)

In [4]:
# 결측치가 존재하는가?
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150000 entries, 0 to 149999
Data columns (total 2 columns):
 #   Column    Non-Null Count   Dtype 
---  ------    --------------   ----- 
 0   document  149995 non-null  object
 1   label     150000 non-null  int64 
dtypes: int64(1), object(1)
memory usage: 2.3+ MB


In [5]:
# 결측치가 150000개의 데이터 중 5개의 결측치가 관찰
# 5개니까 굉장히 적은 양 -> 제외
df.dropna(inplace=True)

In [6]:
# 리뷰 데이터 중 중복된 문장이 존재하는가? -> 확인?? -> value_counts()
df['document'].value_counts()

document
굿                                                181
good                                              92
최고                                                85
쓰레기                                               79
별로                                                66
                                                ... 
굿바이 레닌 표절인것은 이해하는데 왜 뒤로 갈수록 재미없어지냐                 1
이건 정말 깨알 캐스팅과 질퍽하지않은 산뜻한 내용구성이 잘 버무러진 깨알일드!!♥      1
약탈자를 위한 변명, 이라. 저놈들은 착한놈들 절대 아닌걸요.                 1
나름 심오한 뜻도 있는 듯. 그냥 학생이 선생과 놀아나는 영화는 절대 아님          1
흠...포스터보고 초딩영화줄....오버연기조차 가볍지 않구나                  1
Name: count, Length: 146182, dtype: int64

In [7]:
# 중복으로 만들어져있는 리뷰 문자들은 제외 -> 과적합 방지
df.drop_duplicates('document', inplace = True)

In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 146182 entries, 0 to 149999
Data columns (total 2 columns):
 #   Column    Non-Null Count   Dtype 
---  ------    --------------   ----- 
 0   document  146182 non-null  object
 1   label     146182 non-null  int64 
dtypes: int64(1), object(1)
memory usage: 3.3+ MB


In [9]:
# 학습 데이터와 검증 데이터로 데이터를 분할
X = df['document'].values
Y = df['label'].values

# train test 셋으로 분할(분류 데이터 -> label의 비율을 유지)
X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, test_size=0.2, stratify=Y, random_state=42
)

In [10]:
# X의 데이터가 문자임으로 토큰화 작업
komoran = Komoran()
# 사용할 품사 선택
allow_pos = ['NNP', 'NNG', 'VV', 'VA', 'SL', 'MAG']
# 사용하지 않을 단어를 선택
stop_word = ['하다', '되다']
# 글자 수 제한
len_word = 2
# 토큰화 함수 정의
def tokenize(text):
    # 결과를 리스트에 되돌려주기 위해 빈 리스트를 생성
    tokens = []
    for word, pos in komoran.pos(text):
        # 조건 1 : 품사에 포함되어있다면
        # 조건 2 : 금지어에 포함되어있지 않다면
        # 조건 3 : 문자의 길이가 len_word보다 크거나 같다면
        if pos in allow_pos and word not in stop_word and len(word) >= len_word:
            # 3개의 조건을 모두 만족하는 단어를 tokens에 추가
            tokens.append(word)
    return tokens

In [11]:
# 벡터화 객체 생성 (단어의 중요도를 판단하는 벡터화 class 로드)
# 벡터화(자연어 데이터에서 사용하는 스케일링 비슷한 작업)
vectorizer = TfidfVectorizer(
    tokenizer=tokenize,
    ngram_range=(1, 1),
    min_df=3,    # 3회 이상 나온 단어들을 기준으로 중요도 판단
    lowercase=False
)

In [12]:
# train 데이터를 이용하여 fit_transform()
# test 데이터는 transform() -> 데이터의 누수 방지

In [13]:
X_train_vec = vectorizer.fit_transform(X_train)
print(len(vectorizer.get_feature_names_out()))

c:\Users\student\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\feature_extraction\text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


13575


In [20]:
# vectorizer.fit_transform(X_test)
# print(len(vectorizer.get_feature_names_out()))

In [14]:
X_test_vec = vectorizer.transform(X_test)
print(len(vectorizer.get_feature_names_out()))

13575


- train, test에 모두 fit을 하게 되면 두 개의 데이터에서 단어의 추출이 다른 값들을 보인다.(데이터의 누수)
- train을 이용하여 fit을 하고 test데이터는 transform 작업

In [22]:
X_train_vec.toarray()

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]], shape=(116945, 13575))

In [15]:
# X_train_vec와 Y_train 데이터를 이용하여 모델에 학습
model = LinearSVC(C = 1.0)

In [16]:
# 모델에 학습 -> 13575개의 컬럼에서 label 0, 1 사이의 규칙을 찾아내는 과정
model.fit(X_train_vec, Y_train)

,penalty,'l2'
,loss,'squared_hinge'
,dual,'auto'
,tol,0.0001
,C,1.0
,multi_class,'ovr'
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,verbose,0
,random_state,None


In [17]:
pred = model.predict(X_test_vec)

In [18]:
pred

array([1, 0, 0, ..., 0, 1, 1], shape=(29237,))

In [19]:
acc = accuracy_score(pred, Y_test)
f1 = f1_score(pred, Y_test)
print(f'정확도 : {round(acc, 4)}, F1score : {round(f1, 4)}')

정확도 : 0.7735, F1score : 0.7669


In [20]:
# 모델의 성능을 올리기 위해 최적의 파라미터를 찾는 과정
# Pipeline, GridsearchCV, StratifiedKFold
# 파이프라인 생성
pipe = Pipeline(
    [
        (
        'vectorizer', TfidfVectorizer(
            # 고정으로 사용할 매개변수의 값을 지정
            lowercase=False,
            max_df=0.95,   # 너무 자주 등장하는 단어는 배제
            sublinear_tf=True   # tf를 log(1 + tf)로 스케일
            )
        ),
        (
            'clf', LinearSVC()
        )
    ]
)

In [21]:
# pipe에서 사용할 매개변수의 값들을 지정
param_grid = {
    'vectorizer__ngram_range' : [(1, 1), (1, 2)],
    'vectorizer__min_df' : [3, 5],
    'clf__C' : [0.9, 1.0]
}

In [22]:
# 교차 폴드화
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

In [23]:
grid = GridSearchCV(
    estimator=pipe,
    param_grid=param_grid,
    scoring='f1_macro',
    cv=cv,
    n_jobs=-1,
    verbose=1
)

In [24]:
# Gridsearch을 통한 학습
grid.fit(X_train, Y_train)

Fitting 3 folds for each of 8 candidates, totalling 24 fits


,estimator,Pipeline(step...LinearSVC())])
,param_grid,"{'clf__C': [0.9, 1.0], 'vectorizer__min_df': [3, 5], 'vectorizer__ngram_range': [(1, ...), (1, ...)]}"
,scoring,'f1_macro'
,n_jobs,-1
,refit,True
,cv,StratifiedKFo... shuffle=True)
,verbose,1
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,input,'content'


In [25]:
print("최적의 매개변수 :", grid.best_params_)
print("최적의 F1Score :", round(grid.best_score_, 4))

최적의 매개변수 : {'clf__C': 0.9, 'vectorizer__min_df': 3, 'vectorizer__ngram_range': (1, 2)}
최적의 F1Score : 0.7934


### 실습 문제
- 토큰화
    - komoran 사용
    - 품사는 ('NNP', 'NNG', 'VV', 'VA', 'MAG', 'SL')만 사용
    - 단어의 길이는 2 이상
    - 금지어 ('하다', '되다')
- 벡터화
    - TfidVectorizer를 사용
    - 고정 매개변수는
        - tokenizer = komoran 사용
        - lowercase = False
        - max_df = 0.95
        - sublinear_tf = True
- 분류 모델
    - LogisticRegression
    - 고정 매개변수
        - n_jobs = -1
        - class_weight = 'balanced'

- 파이프라인 생성
    - 벡터화 ('vector')
    - 분류 모델 ('clf')

- 교차 검증
    - StratifiedKFold
        - 폴드의 개수는 4
        - shuffle = True
        - random_state = 42

- gridsearch
    - 벡터화
        - ngram_range는 (1, 1), (1, 2)
        - min_df는 3, 5
    - 분류 모델
        - max_iter를 800, 1000
        - C를 1.0, 2.0
- 최적의 파라미터를 찾는다.

In [26]:
# Logistic 모델을 로드
from sklearn.linear_model import LogisticRegression

In [ ]:
# komoran 생성
komoran = Komoran()

allow_pos = ['NNP', 'NNG', 'VV', 'VA', 'MAG', 'SL']
stop_word = ['되다', '하다']

# 토큰화 함수
def tokenize(text):
    # 필터링 된 단어들을 저장할 공간
    tokens = []
    for word, pos in komoran.pos(text):
        # word : komoran을 이용해서 만들어진 단어
        # pos : 형태(품사)
        if pos in allow_pos and word not in stop_word and len(word) >= 2:
            # 위의 조건식 3개를 모두 만족하는 단어를 tokens에 추가
            tokens.append(word)
    return tokens

# 벡터화 생성
vectorizer_komoran = TfidfVectorizer(
    tokenizer = tokenize,
    max_df = 0.95,
    lowercase = False,
    sublinear_tf = True
)
# 모델 정의
model_logi = LogisticRegression(
    n_jobs=-1,
    class_weight='balanced'
)
# 파이프 정의
pipe_logi = Pipeline(
    [ ('vector', vectorizer_komoran), ('clf', model_logi) ]
)

# 교차 검증(계층화 교차 검증) (4회)
cv = StratifiedKFold(n_splits=4, shuffle=True, random_state=42)

# gridsearch에서 사용할 파라미터 값들
# pipe에서 생성한 key값들과 매개변수의 명으로 파라미터 값들을 지정
# {pipe_key}__{매개변수명}
param_grid = {
    # pipe에서 vector의 key를 가진 객체를 찾아서 매개변수 ngram_range 값을 변경
    'vector__ngram_range' : [(1, 1), (1, 2)],
    'vector__min_df' : [3, 5],
    'clf__max_iter' : [800, 1000],
    'clf__C' : [1.0, 2.0]
}

grid_logi = GridSearchCV(
    estimator=pipe_logi,
    param_grid=param_grid,
    scoring='f1_macro',
    verbose=1,
    cv = cv
    # n_jobs -1을 사용했을때 remote error가 뜨면 해당 매개변수를 제거
    # n_jobs=-1
)

In [32]:
grid_logi.fit(X_train, Y_train)

Fitting 4 folds for each of 16 candidates, totalling 64 fits


c:\Users\student\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\feature_extraction\text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
c:\Users\student\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\feature_extraction\text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
c:\Users\student\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\feature_extraction\text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
c:\Users\student\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\feature_extraction\text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
c:\Users\student\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\feature_extraction\text.p

,estimator,Pipeline(step... n_jobs=-1))])
,param_grid,"{'clf__C': [1.0, 2.0], 'clf__max_iter': [800, 1000], 'vector__min_df': [3, 5], 'vector__ngram_range': [(1, ...), (1, ...)]}"
,scoring,'f1_macro'
,n_jobs,None
,refit,True
,cv,StratifiedKFo... shuffle=True)
,verbose,1
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,input,'content'


In [33]:
print(grid_logi.best_params_)
print(grid_logi.best_score_)

{'clf__C': 1.0, 'clf__max_iter': 800, 'vector__min_df': 3, 'vector__ngram_range': (1, 2)}
0.7832177099066779


In [34]:
# gridsearch 모델 중 점수가 가장 높은 모델을 변수에 저장
best_model = grid_logi.best_estimator_

In [35]:
# test data를 이용하여 정확도, f1 score 확인
best_pred = best_model.predict(X_test)

In [36]:
best_acc = accuracy_score(best_pred, Y_test)
best_f1 = f1_score(best_pred, Y_test)

In [37]:
print(round(best_acc, 4))
print(round(best_f1, 4))

0.7898
0.7825


In [38]:
# ratings_test.txt 파일을 로드
df_test = pd.read_csv('../data/ratings_test.txt', sep='\t')

In [39]:
df_test.dropna(inplace=True)

In [40]:
test_x = df_test['document'].values
test_y = df_test['label'].values

In [41]:
best_pred_test = best_model.predict(test_x)

In [42]:
print(round(accuracy_score(best_pred_test, test_y), 4))
print(round(f1_score(best_pred_test, test_y), 4))

0.784
0.7776
